In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

RAW_DATA_DIR = Path("../data/raw/")
PROCESSED_DATA_DIR = Path("../data/processed/")
MODELS_DIR = Path("../models/")

print("Raw data folder contents:")
for f in RAW_DATA_DIR.iterdir():
    print(" -", f.name)

In [2]:
def load_any_file(filepath):
    """
    Handles both plain JSON (a list of records, or a dict with a 'cases' key)
    and JSONL (one JSON object per line).
    """
    records = []
    text = filepath.read_text(encoding="utf-8")
    try:
        data = json.loads(text)
        if isinstance(data, list):
            records = data
        elif isinstance(data, dict) and "cases" in data:
            records = data["cases"]
        else:
            records = [data]
    except json.JSONDecodeError:
        for line in text.splitlines():
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def detect_schema(record):
    """
    Detects which known schema a record follows.
    Now handles THREE known formats.
    """
    if "case_id" in record and "target_available" in record:
        return "ml_ready"
    elif "normalized_case" in record:
        return "raw_normalized_wrapped"
    elif "case" in record and "case_timeline" in record:
        return "raw_normalized_flat"
    else:
        return "unknown"


def normalize_ml_ready(record):
    """Already flat — no changes needed."""
    return record


def _extract_container(record, schema_type):
    """
    Returns the dict that holds 'case', 'case_timeline', 'acts_and_sections'
    regardless of whether it's wrapped under 'normalized_case' or sits at top level.
    """
    if schema_type == "raw_normalized_wrapped":
        return record.get("normalized_case", {})
    else:  # raw_normalized_flat
        return record


def normalize_raw_scrape(record, schema_type):
    """
    Flattens either raw-normalized variant into the same ML-ready shape.
    """
    container = _extract_container(record, schema_type)
    case = container.get("case", {})
    timeline = container.get("case_timeline", [])
    acts = container.get("acts_and_sections", [])

    case_status = case.get("case_status", "") or ""
    is_disposed = "dispos" in case_status.lower()

    disposal_date = None
    days_to_disposal = None
    if is_disposed and timeline:
        disposal_date = timeline[0].get("hearing_date")

    filing_date = case.get("filing_date")
    if is_disposed and disposal_date and filing_date:
        try:
            days_to_disposal = (
                pd.to_datetime(disposal_date) - pd.to_datetime(filing_date)
            ).days
        except Exception:
            days_to_disposal = None

    # case_id: prefer top-level record's own case_id/cnr_number if present, else fall back to case.cnr_number
    case_id = record.get("case_id") or case.get("cnr_number")

    return {
        "case_id": case_id,
        "case_type": case.get("case_type"),
        "court_id": case.get("court_id"),
        "court_id_source": f"FROM_{schema_type.upper()}",
        "court_name": case.get("court_name"),
        "court_type": None,
        "district": case.get("court_district"),
        "state": case.get("court_state"),
        "filing_date": filing_date,
        "registration_date": case.get("registration_date"),
        "first_hearing_date": case.get("first_hearing_date"),
        "next_hearing_date": case.get("next_hearing_date"),
        "current_status": case.get("current_case_stage"),
        "disposal_date": disposal_date,
        "days_to_disposal": days_to_disposal,
        "target_available": bool(is_disposed and days_to_disposal is not None),
        "judge_position": case.get("presiding_judge"),
        "judge_id": None,
        "acts_and_sections": [
            {"act_name": a.get("act"), "section": a.get("section")} for a in acts
        ],
        "is_bailable": None,
        "number_of_sections_charged": None,
        "number_of_section_references": len(acts),
        "case_status_history": [
            {
                "case_id": case_id,
                "status": t.get("purpose_of_hearing"),
                "hearing_date": t.get("hearing_date"),
                "recorded_at": t.get("business_on_date"),
                "judge_position": t.get("judge"),
                "raw_timeline_entry": t,
            }
            for t in timeline
        ],
        "data_source": schema_type.upper(),
        "source_record_identifier": record.get("source", {}).get("source_record_identifier") or case_id,
        "pdf_document_count": 0,
    }


print("All functions defined successfully (now handling 3 schemas).")

All functions defined successfully (now handling 3 schemas).


In [3]:
all_files = list(RAW_DATA_DIR.glob("*.json")) + list(RAW_DATA_DIR.glob("*.jsonl"))
print(f"Found {len(all_files)} files to process\n")

all_normalized = []
unknown_records = []

for filepath in all_files:
    records = load_any_file(filepath)
    file_normalized_count = 0
    for r in records:
        schema = detect_schema(r)
        if schema == "ml_ready":
            all_normalized.append(normalize_ml_ready(r))
            file_normalized_count += 1
        elif schema in ("raw_normalized_wrapped", "raw_normalized_flat"):
            all_normalized.append(normalize_raw_scrape(r, schema))
            file_normalized_count += 1
        else:
            unknown_records.append((filepath.name, r))
    print(f"{filepath.name}: {len(records)} records → {file_normalized_count} normalized")

print(f"\nTotal normalized records: {len(all_normalized)}")
print(f"Unknown/unrecognized records: {len(unknown_records)}")

In [4]:
if unknown_records:
    print("⚠️  Files with unrecognized schema:")
    seen_files = set()
    for fname, r in unknown_records:
        if fname not in seen_files:
            print(f"\n--- {fname} ---")
            print("Top-level fields found:", list(r.keys())[:15])
            seen_files.add(fname)
else:
    print("✅ All records recognized — safe to proceed.")

✅ All records recognized — safe to proceed.


In [5]:
# Convert the list of normalized dictionaries into a single flat table
df = pd.json_normalize(all_normalized, sep="_")

before = len(df)
df = df.drop_duplicates(subset="case_id", keep="first")
after = len(df)
print(f"Combined into {before} rows, removed {before - after} duplicates → {after} unique cases")

# Save the cleaned result so you don't have to redo this every time
output_path = PROCESSED_DATA_DIR / "all_cases_normalized.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Combined into 263 rows, removed 146 duplicates → 117 unique cases
Saved to ..\data\processed\all_cases_normalized.csv


In [6]:
print("=== Target Variable Availability (for Days-to-Disposal model) ===")
print(df["target_available"].value_counts())
print(f"\nDisposed cases usable for disposal-time training: {df['target_available'].sum()}")
print(f"Pending cases (no disposal target, but usable for next-hearing training): {(~df['target_available']).sum()}")

print("\n=== Missing Values in Key Columns ===")
key_cols = ["case_type", "district", "state", "filing_date", "court_name"]
print(df[key_cols].isna().sum())

print("\n=== Case Type Distribution ===")
print(df["case_type"].value_counts())

print("\n=== Geography Coverage ===")
print("Districts:", df["district"].nunique(), "unique")
print("States:", df["state"].nunique(), "unique")
print(df["state"].value_counts())

=== Target Variable Availability (for Days-to-Disposal model) ===
target_available
False    112
True       5
Name: count, dtype: int64

Disposed cases usable for disposal-time training: 5
Pending cases (no disposal target, but usable for next-hearing training): 112

=== Missing Values in Key Columns ===
case_type       0
district       23
state           0
filing_date     0
court_name      1
dtype: int64

=== Case Type Distribution ===
case_type
Civil M.A. - Civil Misc. Application                         24
Cri.Bail Appln. - Bail Application                           11
R.C.S. - Regular Civil Suit                                  11
Cri.Appeal - Criminal Appeal                                 10
Marriage Petn. - Marriage Petition                           10
Cri.M.A. - Criminal Misc. Application                        10
R.C.C. - Regular Criminal Case                               10
Elec.Petn. - Election Petition                                4
BAIL APPLICATION                      

In [7]:
# ============================================
# CELL 7: Unpack case_status_history into a training table
# for the Days-to-Next-Hearing model
# ============================================

# Load the saved normalized data (fresh read, so this cell works independently too)
df = pd.read_csv(PROCESSED_DATA_DIR / "all_cases_normalized.csv")

# case_status_history was saved as a string representation of a list — need to parse it back
import ast

def safe_parse_history(val):
    if pd.isna(val) or val == "":
        return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []

df["case_status_history_parsed"] = df["case_status_history"].apply(safe_parse_history)

# Unpack into a proper history table (like your CASE_STATUS_HISTORY ER entity)
history_rows = []
for _, row in df.iterrows():
    for entry in row["case_status_history_parsed"]:
        history_rows.append({
            "case_id": row["case_id"],
            "case_type": row["case_type"],
            "court_name": row["court_name"],
            "district": row["district"],
            "state": row["state"],
            "hearing_date": entry.get("hearing_date"),
            "business_on_date": entry.get("recorded_at"),
            "status": entry.get("status"),
        })

history_df = pd.DataFrame(history_rows)
history_df["hearing_date"] = pd.to_datetime(history_df["hearing_date"], errors="coerce")
history_df["business_on_date"] = pd.to_datetime(history_df["business_on_date"], errors="coerce")

# Drop rows where dates couldn't be parsed
history_df = history_df.dropna(subset=["hearing_date", "business_on_date"])

# TARGET VARIABLE for Model 1
history_df["days_to_next_hearing"] = (
    history_df["hearing_date"] - history_df["business_on_date"]
).dt.days

# Remove impossible negative values (data errors)
history_df = history_df[history_df["days_to_next_hearing"] >= 0]

print(f"Total training rows for Days-to-Next-Hearing model: {len(history_df)}")
print(history_df["days_to_next_hearing"].describe())

Total training rows for Days-to-Next-Hearing model: 786
count    786.000000
mean      26.456743
std       22.743461
min        0.000000
25%        9.000000
50%       22.000000
75%       37.000000
max      116.000000
Name: days_to_next_hearing, dtype: float64


In [8]:
print("df exists:", "df" in dir())
print("history_df exists:", "history_df" in dir())
if "history_df" in dir():
    print("history_df shape:", history_df.shape)
    print(history_df.head())

In [9]:
# ============================================
# CELL 9: Feature Engineering for Days-to-Next-Hearing Model
# ============================================

model_df = history_df.copy()

# Historical pace features — average delay per case_type and per court
# (Using the full dataset for now since our sample is small; once data grows,
#  we'll compute this from TRAIN split only to avoid leakage)
case_type_pace = model_df.groupby("case_type")["days_to_next_hearing"].transform("mean")
court_pace = model_df.groupby("court_name")["days_to_next_hearing"].transform("mean")

model_df["case_type_historical_pace"] = case_type_pace
model_df["court_historical_pace"] = court_pace

# One-hot encode categorical features
categorical_cols = ["case_type", "court_name", "district", "state"]
model_df_encoded = pd.get_dummies(model_df, columns=categorical_cols, drop_first=True)

print(f"Feature matrix shape: {model_df_encoded.shape}")
print("Columns:", model_df_encoded.columns.tolist()[:20], "...")

Feature matrix shape: (786, 28)
Columns: ['case_id', 'hearing_date', 'business_on_date', 'status', 'days_to_next_hearing', 'case_type_historical_pace', 'court_historical_pace', 'case_type_Civil Appeal PPE - Civil Appeal under Public Premises Act', 'case_type_Civil M.A. - Civil Misc. Application', 'case_type_Civil Suit', 'case_type_Commercial Suit', 'case_type_Cri.Appeal - Criminal Appeal', 'case_type_Cri.Bail Appln. - Bail Application', 'case_type_Cri.M.A. - Criminal Misc. Application', 'case_type_Cri.Rev.App. - Criminal Revision Application', 'case_type_Elec.Petn. - Election Petition', 'case_type_M.C.A. - Misc.Civil Appeal', 'case_type_Marriage Petn. - Marriage Petition', 'case_type_PWDVA Appln. - Application under Domestic Violence Act', 'case_type_R.C.A. - Regular Civil Appeal'] ...


In [10]:
# ============================================
# CELL 10: Prepare final features (X) and target (y), then split
# ============================================

from sklearn.model_selection import train_test_split

# Drop columns that aren't real predictive features
drop_cols = ["case_id", "hearing_date", "business_on_date", "status", "days_to_next_hearing"]
X = model_df_encoded.drop(columns=drop_cols)
y = model_df_encoded["days_to_next_hearing"]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")

X shape: (786, 23)
y shape: (786,)

Training set: 628 rows
Test set: 158 rows


In [11]:
# ============================================
# CELL 11: Train baseline Random Forest model
# ============================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"MAE (Mean Absolute Error): {mae:.2f} days")
print(f"RMSE (Root Mean Squared Error): {rmse:.2f} days")

MAE (Mean Absolute Error): 15.48 days
RMSE (Root Mean Squared Error): 21.81 days


In [12]:
# ============================================
# CELL 12: XGBoost model (upgrade from baseline)
# ============================================
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.08,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))

print(f"XGBoost MAE: {xgb_mae:.2f} days")
print(f"XGBoost RMSE: {xgb_rmse:.2f} days")
print(f"\nCompare to baseline — MAE: 15.48, RMSE: 21.81")

XGBoost MAE: 15.56 days
XGBoost RMSE: 21.89 days

Compare to baseline — MAE: 15.48, RMSE: 21.81


In [13]:
# ============================================
# CELL 12b: Pick the winning model automatically
# ============================================
if xgb_mae <= mae:
    final_model = xgb_model
    final_model_name = "xgboost"
    print(f"Using XGBoost (MAE {xgb_mae:.2f} vs RF {mae:.2f})")
else:
    final_model = model  # the RandomForestRegressor from Cell 11
    final_model_name = "random_forest"
    print(f"Using Random Forest (MAE {mae:.2f} vs XGBoost {xgb_mae:.2f})")

Using Random Forest (MAE 15.48 vs XGBoost 15.56)


In [14]:
# ============================================
# CELL 13: Confidence scoring
# ============================================
final_preds = final_model.predict(X_test)
residuals = y_test.values - final_preds
resid_std = residuals.std()

print(f"Residual std dev: {resid_std:.2f} days")

def get_confidence_range(prediction, std=resid_std):
    lower = max(0, prediction - 1.28 * std)
    upper = prediction + 1.28 * std
    width = upper - lower
    if width < 20:
        label = "high"
    elif width < 45:
        label = "medium"
    else:
        label = "low"
    return round(lower), round(upper), label

test_pred = final_preds[0]
lo, hi, conf = get_confidence_range(test_pred)
print(f"\nExample: prediction={test_pred:.1f}, range=({lo}-{hi}), confidence={conf}")

Residual std dev: 21.81 days

Example: prediction=16.0, range=(0-44), confidence=medium


In [15]:
# ============================================
# CELL 14: SHAP explainability
# ============================================
import shap

explainer = shap.TreeExplainer(final_model)

def explain_prediction(X_row, feature_names, top_n=3):
    row_2d = X_row.values.reshape(1, -1) if hasattr(X_row, "values") else X_row.reshape(1, -1)
    shap_vals = explainer.shap_values(row_2d)
    if isinstance(shap_vals, list):  # some models return a list
        shap_vals = shap_vals[0]
    shap_vals = shap_vals[0] if shap_vals.ndim > 1 else shap_vals
    contributions = list(zip(feature_names, shap_vals))
    contributions.sort(key=lambda x: abs(x[1]), reverse=True)
    top = contributions[:top_n]
    explanation = []
    for feat, val in top:
        direction = "increased" if val > 0 else "decreased"
        clean_name = feat.replace("_", " ")
        explanation.append({
            "feature": feat,
            "contribution_days": round(float(val), 1),
            "text": f"{clean_name} {direction} the estimate by about {abs(round(val))} days"
        })
    return explanation

sample_explanation = explain_prediction(X_test.iloc[0], X.columns.tolist())
print("Sample explanation for one prediction:")
for e in sample_explanation:
    print(" -", e["text"])

Sample explanation for one prediction:
 - case type historical pace decreased the estimate by about 10 days
 - case type Cri.Bail Appln. - Bail Application increased the estimate by about 0 days
 - court name JMFC COURT KHED increased the estimate by about 0 days


In [16]:
# ============================================
# CELL 15: Save model + supporting artifacts
# ============================================
import joblib
from pathlib import Path

# Safety check — make sure MODELS_DIR exists (this is likely what broke before)
if "MODELS_DIR" not in dir():
    MODELS_DIR = Path("../models")
MODELS_DIR = Path(MODELS_DIR)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Using MODELS_DIR:", MODELS_DIR.resolve())

# Save the trained model — branches based on which model won in Cell 12b
if final_model_name == "xgboost":
    final_model.save_model(str(MODELS_DIR / "delay_next_hearing_model.json"))
else:
    joblib.dump(final_model, MODELS_DIR / "delay_next_hearing_model.pkl")

# Save which model type was used (so the API loader knows how to reload it)
joblib.dump(final_model_name, MODELS_DIR / "model_type.pkl")

# Save the exact column order X was trained on (CRITICAL for future predictions to align correctly)
feature_columns = X.columns.tolist()
joblib.dump(feature_columns, MODELS_DIR / "feature_columns.pkl")

# Save the historical pace lookup tables (needed to featurize a brand NEW case later)
case_type_pace_lookup = model_df.groupby("case_type")["days_to_next_hearing"].mean().to_dict()
court_pace_lookup = model_df.groupby("court_name")["days_to_next_hearing"].mean().to_dict()
overall_mean_pace = model_df["days_to_next_hearing"].mean()  # fallback for unseen case_type/court

joblib.dump(case_type_pace_lookup, MODELS_DIR / "case_type_pace_lookup.pkl")
joblib.dump(court_pace_lookup, MODELS_DIR / "court_pace_lookup.pkl")
joblib.dump(overall_mean_pace, MODELS_DIR / "overall_mean_pace.pkl")
joblib.dump(resid_std, MODELS_DIR / "resid_std.pkl")

print("Saved model and all supporting files to:", MODELS_DIR)
print("Files:", list(MODELS_DIR.iterdir()))

Using MODELS_DIR: C:\Users\Admin\nyaymitra-ml\models
Saved model and all supporting files to: ..\models
Files: [WindowsPath('../models/case_type_pace_lookup.pkl'), WindowsPath('../models/court_pace_lookup.pkl'), WindowsPath('../models/delay_next_hearing_model.pkl'), WindowsPath('../models/feature_columns.pkl'), WindowsPath('../models/model_type.pkl'), WindowsPath('../models/overall_mean_pace.pkl'), WindowsPath('../models/resid_std.pkl')]


In [20]:
# ============================================
# CELL 16: Full prediction pipeline for a NEW case (not from training data)
# ============================================

def build_feature_row(case_type, court, district, state):
    """Takes raw case details and builds a feature row matching training columns."""
    row = {col: 0 for col in feature_columns}
    
    row["case_type_historical_pace"] = case_type_pace_lookup.get(case_type, overall_mean_pace)
    row["court_historical_pace"] = court_pace_lookup.get(court, overall_mean_pace)
    
    for col in [f"case_type_{case_type}", f"court_name_{court}",
                f"district_{district}", f"state_{state}"]:
        if col in row:
            row[col] = 1
    
    return pd.DataFrame([row])[feature_columns]

def predict_delay(case_type, court, district, state):
    X_new = build_feature_row(case_type, court, district, state)
    prediction = final_model.predict(X_new)[0]
    lower, upper, confidence = get_confidence_range(prediction)
    explanation = explain_prediction(X_new.iloc[0], feature_columns)
    
    return {
        "predicted_delay_days": round(float(prediction)),
        "predicted_delay_lower": lower,
        "predicted_delay_upper": upper,
        "confidence_score": round(1 - (upper - lower) / 100, 2),
        "confidence_label": confidence,
        "explanation": explanation,
        "model_version": f"v1_{final_model_name}",
    }

# Test it with a realistic example
result = predict_delay(
    case_type="Cri.Bail Appln. - Bail Application",
    court="District and Sessions Court",
    district="Pune",
    state="Maharashtra"
)
import json
print(json.dumps(result, indent=2))

{
  "predicted_delay_days": 6,
  "predicted_delay_lower": 0,
  "predicted_delay_upper": 34,
  "confidence_score": 0.66,
  "confidence_label": "medium",
  "explanation": [
    {
      "feature": "case_type_historical_pace",
      "contribution_days": -20.5,
      "text": "case type historical pace decreased the estimate by about 20 days"
    },
    {
      "feature": "case_type_Cri.Bail Appln. - Bail Application",
      "contribution_days": -1.0,
      "text": "case type Cri.Bail Appln. - Bail Application decreased the estimate by about 1 days"
    },
    {
      "feature": "district_Pune",
      "contribution_days": 0.6,
      "text": "district Pune increased the estimate by about 1 days"
    }
  ],
  "model_version": "v1_random_forest"
}


In [21]:
# ============================================
# CELL 17: Write the reusable prediction logic to a proper .py file
# ============================================

predict_module_code = '''
import pandas as pd
import joblib
import shap
import xgboost as xgb
from pathlib import Path

MODELS_DIR = Path(__file__).parent.parent / "models"

model_type = joblib.load(MODELS_DIR / "model_type.pkl")

if model_type == "xgboost":
    model = xgb.XGBRegressor()
    model.load_model(str(MODELS_DIR / "delay_next_hearing_model.json"))
else:
    model = joblib.load(MODELS_DIR / "delay_next_hearing_model.pkl")

feature_columns = joblib.load(MODELS_DIR / "feature_columns.pkl")
case_type_pace_lookup = joblib.load(MODELS_DIR / "case_type_pace_lookup.pkl")
court_pace_lookup = joblib.load(MODELS_DIR / "court_pace_lookup.pkl")
overall_mean_pace = joblib.load(MODELS_DIR / "overall_mean_pace.pkl")
resid_std = joblib.load(MODELS_DIR / "resid_std.pkl")

explainer = shap.TreeExplainer(model)


def get_confidence_range(prediction, std=resid_std):
    lower = max(0, prediction - 1.28 * std)
    upper = prediction + 1.28 * std
    width = upper - lower
    if width < 20:
        label = "high"
    elif width < 45:
        label = "medium"
    else:
        label = "low"
    return round(lower), round(upper), label


def build_feature_row(case_type, court, district, state):
    row = {col: 0 for col in feature_columns}
    row["case_type_historical_pace"] = case_type_pace_lookup.get(case_type, overall_mean_pace)
    row["court_historical_pace"] = court_pace_lookup.get(court, overall_mean_pace)

    for col in [f"case_type_{case_type}", f"court_name_{court}",
                f"district_{district}", f"state_{state}"]:
        if col in row:
            row[col] = 1

    return pd.DataFrame([row])[feature_columns]


def explain_prediction(X_row, top_n=3):
    row_2d = X_row.values.reshape(1, -1)
    shap_vals = explainer.shap_values(row_2d)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[0]
    shap_vals = shap_vals[0] if shap_vals.ndim > 1 else shap_vals

    contributions = list(zip(feature_columns, shap_vals))
    contributions.sort(key=lambda x: abs(x[1]), reverse=True)
    top = contributions[:top_n]
    explanation = []
    for feat, val in top:
        direction = "increased" if val > 0 else "decreased"
        clean_name = feat.replace("_", " ")
        explanation.append({
            "feature": feat,
            "contribution_days": round(float(val), 1),
            "text": f"{clean_name} {direction} the estimate by about {abs(round(val))} days"
        })
    return explanation


def predict_delay(case_type, court, district, state):
    X_new = build_feature_row(case_type, court, district, state)
    prediction = model.predict(X_new)[0]
    lower, upper, confidence = get_confidence_range(prediction)
    explanation = explain_prediction(X_new.iloc[0])

    return {
        "predicted_delay_days": round(float(prediction)),
        "predicted_delay_lower": lower,
        "predicted_delay_upper": upper,
        "confidence_score": round(1 - (upper - lower) / 100, 2),
        "confidence_label": confidence,
        "explanation": explanation,
        "model_version": f"v1_{model_type}",
    }
'''

src_dir = Path("../src")
src_dir.mkdir(exist_ok=True)
(src_dir / "delay_predictor.py").write_text(predict_module_code)

print("Saved to ../src/delay_predictor.py")

Saved to ../src/delay_predictor.py


In [22]:
# ============================================
# CELL 18: Write the FastAPI app
# ============================================

api_code = '''
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded

from delay_predictor import predict_delay

app = FastAPI(title="NyayMitra Delay Prediction Service")

limiter = Limiter(key_func=get_remote_address)
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)


class CaseInput(BaseModel):
    case_type: str
    court: str
    district: str
    state: str


@app.post("/predict-delay")
@limiter.limit("30/minute")
def predict_delay_endpoint(request: Request, case: CaseInput):
    try:
         result = predict_delay(case.case_type, case.court, case.district, case.state)
        return result
    except Exception as e:
        print(f"Internal error: {e}")
        return JSONResponse(status_code=500, content={"error": "Could not generate a prediction for this case."})


@app.get("/health")
def health():
    return {"status": "ok"}
'''

(src_dir / "api.py").write_text(api_code)
print("Saved to ../src/api.py")
print("\\nTo run the API, open a terminal and run:")
print("cd src && uvicorn api:app --reload --port 8001")

Saved to ../src/api.py
\nTo run the API, open a terminal and run:
cd src && uvicorn api:app --reload --port 8001
